# 1. From a pile of text to a table

**The question this notebook answers: what is one row?**

That sounds like a formality. It is not. Almost every mistake later in this course traces
back to a row meaning something different from what you assumed — a claim about *people*
tested on a table of *messages*, an average over a unit nobody cares about, a count that
silently double-counts.

So before any plot, two jobs:

1. **Get the data into rows that match the question.** Here that means a regular expression,
   because the raw material is a wall of text.
2. **Add what the question needs and the data does not have.** Enrichment. This is where
   most analyses are actually won, and it is the part people skip because it feels like
   admin.

We do both on a showcase corpus first, where you can check your work against something
known, and then on your own chat.

In [ ]:
import re

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from goad_toolkit.datatransforms import Pipeline, TimeFeatures, TransformBase
from loguru import logger

from wa_analyzer.data import PROCESSED, load_own_chat, load_showcase

## 1.1 The showcase: five years of Ubuntu IRC

Two public IRC channels, `#ubuntu-uk` and `#ubuntu-nl`, from 2013 to 2017. Real conversation
between real people, with the same awkward shape your own chat export has.

It ships with this repo, so this runs offline.

In [ ]:
irc = load_showcase("ubuntu_irc")
print(irc.shape)
irc.head()

Three columns and 3,371 rows. So one row is... what?

Look at the `text` column before assuming.

In [ ]:
day = irc.iloc[900]
print(f"{day.channel}  {day.created.date()}\n")
print(day.text[:400])

**One row is a whole day of one channel**, with every message of that day packed into a
single string.

That is not a useful unit for anything we want to ask. "How long are messages?" "Who talks
most?" "When is the channel busy?" — every one of those needs *one row per message*.

Nothing is broken here. The data was simply stored for a different purpose than ours. This
is normal, and noticing it is the job.

## 1.2 Regular expressions: pulling structure out of text

Every line looks like this:

```
[00:08] <mapp> yay
```

Three things we want — a time, an author, a message — and some punctuation holding them
apart. A regular expression is a way of describing that pattern precisely enough for the
computer to pull the pieces out.

Let us build it against one line.

In [ ]:
line = "[00:00] <neuro> yay, it's officially pay day! (for me)"

pattern = re.compile(r"^\[(\d{2}):(\d{2})\]\s+<(\S+)>\s+(.*)$")
pattern.match(line).groups()

Four groups back, in order. Reading the pattern piece by piece:

| piece | means |
|---|---|
| `^` | start of the line — so a `[` in the middle of a sentence cannot match |
| `\[` | a literal `[`. Backslashed, because a bare `[` starts a character class |
| `(\d{2})` | **group**: exactly two digits. `\d` is shorthand for `[0-9]` |
| `:` | a literal colon |
| `\]` | a literal `]` |
| `\s+` | one or more whitespace characters |
| `<(\S+)>` | angle brackets around a **group** of one or more *non*-whitespace characters |
| `(.*)` | **group**: any character, zero or more times — the message itself |
| `$` | end of the line |

The parentheses are what makes this useful. Without them the pattern only answers "does
this line look right?"; with them it hands back the four pieces.

Two habits worth taking from this:

- **Anchor with `^` and `$`** unless you have a reason not to. An unanchored pattern will
  cheerfully find a match in the middle of something you did not mean.
- **`\S+` beats `.*` when you know there is no whitespace.** `.*` is greedy: it takes as much
  as it can and only gives back when forced. For a nickname that is wrong, and the bug it
  causes is subtle.

Now run it over everything. One row per line, per day, per channel.

In [ ]:
rows, unmatched = [], []
for day in irc.itertuples():
    for line in day.text.split("\n"):
        if not line.strip():
            continue
        m = pattern.match(line)
        if m:
            hh, mm, author, message = m.groups()
            rows.append((day.created, day.channel, int(hh), int(mm), author, message))
        else:
            unmatched.append((day.channel, line))

msgs = pd.DataFrame(rows, columns=["date", "channel", "hh", "mm", "author", "message"])
print(f"{len(msgs):,} messages, {len(unmatched):,} lines did not match")
msgs.head()

## 1.3 The part everyone skips: what did *not* match?

615,683 rows is a satisfying number and it is the wrong thing to look at. The interesting
number is the other one.

A regex that matches most lines feels like success. But the lines it missed are not random —
they are the ones that are *shaped differently*, which is exactly where the surprises live.
Never accept a parse without looking at its leftovers.

In [ ]:
missed = pd.DataFrame(unmatched, columns=["channel", "line"])
print(f"{len(missed):,} unmatched lines ({len(missed) / (len(msgs) + len(missed)):.2%})\n")

# Do not just eyeball a sample -- put the leftovers into named buckets.
missed["kind"] = "something else"
missed.loc[missed.line.str.match(r"^\[\d{2}:\d{2}\]\s+\*\s"), "kind"] = "action line"
missed.loc[missed.line.str.contains(r"[\x00-\x08\x0b-\x1f]"), "kind"] = "control characters"
print(missed.kind.value_counts().to_string(), "\n")

for kind, group in missed.groupby("kind"):
    print(f"{kind}:")
    print(f"   {group.line.iloc[0][:88]!r}\n")

Two different things are hiding in there.

**Action lines.** IRC has a `/me` command, and it logs differently — `* nick waves` rather
than `<nick> waves`. Same information, different shape. Our pattern was written for one and
not the other.

**Control characters.** Lines starting with `\x01` and `\x03` are what is left of IRC's
colour and formatting codes, partly stripped when this corpus was built. The message text
survived; the `[HH:MM] <nick>` prefix did not.

The first is worth fixing — those are real messages by real people, and dropping them would
quietly bias anything we measure about who says what. The second is damage, and the honest
move is to count it and move on rather than pretend to recover it.

In [ ]:
action = re.compile(r"^\[(\d{2}):(\d{2})\]\s+\*\s+(\S+)\s+(.*)$")

rows, still_missing = [], []
for day in irc.itertuples():
    for line in day.text.split("\n"):
        if not line.strip():
            continue
        m = pattern.match(line) or action.match(line)
        if m:
            hh, mm, author, message = m.groups()
            rows.append((day.created, day.channel, int(hh), int(mm), author, message,
                         bool(action.match(line))))
        else:
            still_missing.append(line)

msgs = pd.DataFrame(rows, columns=["date", "channel", "hh", "mm",
                                   "author", "message", "is_action"])
coverage = len(msgs) / (len(msgs) + len(still_missing))
print(f"{len(msgs):,} messages, {coverage:.2%} of lines parsed")
print(f"{msgs.is_action.sum():,} of them are action lines")

**Write that number down.** "We recovered 99.94% of lines; the remainder are corrupted
formatting codes" is a sentence that belongs in your report — it is the difference between
an analysis somebody can trust and one they cannot.

## 1.4 Pipelines: build your own, then compile them

The regex work above (1.2, 1.3) built `msgs` once, by hand, cell by cell — worth doing
slowly, because you needed to see the pattern fail and read what it missed before trusting
it. But "here is one file, parse it once" was a property of this lesson, not of the job. A
new export, a new channel, next month's logs: the same parse runs again, the same way, every
time. That is not a one-off, and once you have derived the logic, redoing it by hand each
time is exactly the failure mode a pipeline exists to prevent.

Thirty loose cells doing repeatable work — parsing, enrichment, anything you would want to
rerun — will eventually get run out of order, or on the wrong dataframe, and the honest
answer to "what did you do to the data" becomes "read the notebook".

`goad_toolkit`'s `Pipeline` fixes that by making the sequence an object: named steps, declared
up front, applied in order to a copy. A step is any class with one method — `transform`. That
is the whole contract, and it is why the steps below are as short as they are. By the end of
this section, the parse will be one of those steps too — not just the enrichment that follows
it.

**Some steps you never have to write.** `TimeFeatures` — calendar columns from a timestamp —
already ships in `goad_toolkit`:

In [ ]:
demo = Pipeline()
demo.add(TimeFeatures, column="date")
demo.apply(msgs.head(3))[["date", "day_name", "isoweek", "year_week"]]

Two things happened there and neither needed you to write a class:

- `TimeFeatures` came from `goad_toolkit`, not from this notebook.
- `Pipeline.add` took the *class*, not an instance — the step gets configured and run inside
  `.apply()`. That is what makes a pipeline a description you can print, edit and reuse,
  rather than a list of already-built objects.

Most of what you need to extract will not ship, though — nobody packaged a transform for
"does this IRC message address someone by name". For that you subclass `TransformBase`
yourself. The whole contract is one method:

```python
class MyStep(TransformBase):
    def transform(self, data: pd.DataFrame, column: str) -> pd.DataFrame:
        ...
        return data
```

The base class does the rest: it stores whatever keyword arguments you pass to `pipeline.add`
and hands them to `transform` unpacked, it checks that a `column` argument actually exists in
the frame before your code runs, and it gives the step a name and a `__repr__`. That is the
whole reason `RegexFeature` below reads as four lines of logic and nothing else.

**This is the extendability that matters.** A pipeline step is not a fixed vocabulary — it is
any class shaped like this, whether it shipped with `goad_toolkit` or you wrote it five
minutes ago. Lesson 7 reuses the identical shape (`transform` → `plot`) for chart components.

In [ ]:
class RegexFeature(TransformBase):
    """Add a feature extracted from a text column with a regular expression.

    When `mode="extract"`, logs how many rows matched via `loguru` -- extraction is the
    one mode where "no match" is silent otherwise.
    """

    def transform(
        self,
        data: pd.DataFrame,
        column: str,
        pattern: str,
        feature: str,
        mode: str = "count",
    ) -> pd.DataFrame:
        text = data[column].fillna("")
        if mode == "count":
            data[feature] = text.str.count(pattern)
        elif mode == "has":
            data[feature] = text.str.contains(pattern, regex=True)
        elif mode == "extract":
            data[feature] = text.str.extract(pattern, expand=False)
            matched = data[feature].notna().sum()
            logger.info(
                f"{self.name}: extracted '{feature}' from {matched:,}/{len(data):,} rows "
                f"({matched / len(data):.1%}); {len(data) - matched:,} rows had no match"
            )
        else:
            raise ValueError(f"mode must be count/has/extract, got {mode!r}")
        return data

Two things about `RegexFeature` worth noticing, because both will bite you otherwise.

**The new column is called `feature`, not `name`.** `Pipeline.add(name=...)` already uses
`name` for the *step's* name, and the base class consumes it. Any transform that names an
output column has to call the parameter something else.

**Parameters are spelled out in the signature**, not swallowed by `**kwargs`. That is what
lets the base class check them, and it is what makes the step readable a month from now.

A third thing worth noticing, because it is the point of this section: **`mode="extract"`
reports its own coverage.** `count` and `has` fail loudly if something is wrong — a column of
all zeros or all `False` is visible the moment you look at it. `extract` fails silently: a row
with no match becomes `NaN`, which looks exactly like a row nobody checked. 1.3 caught the
same class of problem for the regex parse by hand, with a `print`. Here the step logs it
itself, via `loguru`, as a side effect of running — so the coverage number shows up every time
the pipeline runs, not just the one time you remembered to add a print for it.

In [ ]:
pipeline = Pipeline()
pipeline.add(TimeFeatures, column="date")
pipeline.add(RegexFeature, name="urls",
             column="message", pattern=r"https?://\S+", feature="has_url", mode="has")
pipeline.add(RegexFeature, name="questions",
             column="message", pattern=r"\?", feature="n_question", mode="count")
pipeline.add(RegexFeature, name="mentions",
             column="message", pattern=r"^(\S+)[:,]\s", feature="addressed_to", mode="extract")

enriched = pipeline.apply(msgs)
enriched.head()

`addressed_to` is the one worth pausing on. On IRC people answer each other by name —
`daftykins: try rebooting`. That single regex turns a flat list of messages into something
with a *reply structure*, which is a different kind of data entirely. Lesson 7 builds a
social graph out of exactly this column.

That is what enrichment means: not tidying, but adding a dimension the raw data did not
have.

And because the pipeline is an object, it can tell you what it did.

In [ ]:
print(pipeline)

> **Your turn, briefly.** Add one more step of your own to the pipeline above. Some ideas:
> the number of capital letters, whether the message ends in a full stop, whether it
> contains a smiley, message length in words. Re-run `pipeline.apply(msgs)` and check the
> column appears.
>
> Keep the ones you find interesting — lesson 5 uses features exactly like these to work out
> what distinguishes one person's writing from another's.

**One more step belongs in here: the parse itself.** 1.2 and 1.3 built `msgs` by hand because
you needed to see the pattern fail and read the leftovers before trusting it. That reasoning
does not survive the lesson, though — a new export, a new channel, next month's logs all need
the identical parse rerun, the same way, every time. So it goes in the pipeline too, wrapping
the exact two regexes from 1.2 and 1.3 into one step.

In [ ]:
class ParseIRCLines(TransformBase):
    """Turn one-row-per-day IRC logs into one row per message.

    Combines the `<nick>` pattern from 1.2 with the `/me` action-line fallback from 1.3, and
    replaces 1.3's manual coverage `print`s with a `loguru` log line.
    """

    LINE = re.compile(r"^\[(\d{2}):(\d{2})\]\s+<(\S+)>\s+(.*)$")
    ACTION = re.compile(r"^\[(\d{2}):(\d{2})\]\s+\*\s+(\S+)\s+(.*)$")

    def transform(self, data: pd.DataFrame, text_column: str = "text") -> pd.DataFrame:
        rows, missing = [], 0
        for row in data.itertuples():
            for line in getattr(row, text_column).split("\n"):
                if not line.strip():
                    continue
                m = self.LINE.match(line) or self.ACTION.match(line)
                if m:
                    hh, mm, author, message = m.groups()
                    rows.append((row.created, row.channel, int(hh), int(mm), author,
                                 message, bool(self.ACTION.match(line))))
                else:
                    missing += 1

        parsed = pd.DataFrame(rows, columns=["date", "channel", "hh", "mm",
                                             "author", "message", "is_action"])
        coverage = len(parsed) / (len(parsed) + missing)
        logger.info(
            f"{self.name}: parsed {len(parsed):,} messages, {coverage:.2%} of lines "
            f"({missing:,} unparsed)"
        )
        return parsed

In [ ]:
full_pipeline = Pipeline()
full_pipeline.add(ParseIRCLines)
full_pipeline.add(TimeFeatures, column="date")
full_pipeline.add(RegexFeature, name="urls",
                   column="message", pattern=r"https?://\S+", feature="has_url", mode="has")
full_pipeline.add(RegexFeature, name="questions",
                   column="message", pattern=r"\?", feature="n_question", mode="count")
full_pipeline.add(RegexFeature, name="mentions",
                   column="message", pattern=r"^(\S+)[:,]\s", feature="addressed_to", mode="extract")

from_scratch = full_pipeline.apply(irc)
assert len(from_scratch) == len(enriched)
from_scratch.head()

Same row count as `enriched`, derived straight from `irc` in one call. That is the corrected
version of "one-off" from the start of this section — nothing here was actually a one-off.
Point `full_pipeline` at a fresh year of logs and every step reruns exactly the way it did on
this one, from the regex parse through to `addressed_to`.

## 1.5 "Cleaned" is a claim somebody else made

The publishers of this corpus say they removed bots. Worth checking before you build on top
of it, because a bot is not a person and every per-author statistic will quietly include it.

Try it **without reading any messages** — structure only. That habit generalises to spam and
abuse, where reading everything is not an option.

In [ ]:
def author_profile(g: pd.DataFrame) -> pd.Series:
    return pd.Series({
        "n": len(g),
        "repeat_rate": 1 - g.message.nunique() / len(g),
        "active_days": g.date.nunique(),
        "busiest_day_share": g.groupby("date").size().max() / len(g),
        "median_length": g.message.str.len().median(),
    })


profiles = enriched.groupby("author").apply(author_profile, include_groups=False)
profiles = profiles[profiles.n >= 20]
print(f"{len(profiles):,} authors with at least 20 messages")

`repeat_rate` is the share of an author's messages that duplicate something they already
said. A script repeats itself constantly; a person, occasionally. So filter on it.

In [ ]:
suspects = profiles[profiles.repeat_rate > 0.4].sort_values("repeat_rate", ascending=False)
print(f"{len(suspects)} authors flagged: {', '.join(suspects.index)}")

Same numbers, now compared instead of listed. The eight highest `repeat_rate`s, with the 0.4
flag threshold marked — blue crossed it, grey did not. `lubotu3\`` is picked out in red,
because it is the one author here that actually is a bot. Watch where its bar lands.

In [ ]:
top = profiles.nlargest(8, "repeat_rate").reset_index()
palette = {
    author: "#c44e52" if author == "lubotu3`" else ("#4c72b0" if rate > 0.4 else "#cccccc")
    for author, rate in zip(top.author, top.repeat_rate)
}

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=top, x="repeat_rate", y="author", hue="author",
            palette=palette, legend=False, ax=ax)
ax.axvline(0.4, color="black", linestyle="--", linewidth=1)
ax.text(0.41, len(top) - 1, "flag threshold", va="center", fontsize=9)
ax.set_xlabel("repeat_rate")
ax.set_ylabel("")
ax.set_title("repeat_rate flags a human ritual and misses the actual bot")
fig.tight_layout()

The biggest of them is `brobostigon` — 9,161 messages over 1,797 days. Look at what they
repeated.

In [ ]:
brobo = enriched[enriched.author == "brobostigon"]
top = brobo.message.value_counts().head(3)
for text, count in top.items():
    days = brobo[brobo.message == text].date.nunique()
    print(f"{count:>5}x  on {days:>5} different days   {text!r}")

**"morning boys and girls."** 1,335 times, on 1,334 different days. Someone saying good
morning to their friends, every morning, for five years.

Now the bot.

In [ ]:
lubotu = enriched[enriched.author == "lubotu3`"]
for text, count in lubotu.message.value_counts().head(3).items():
    print(f"{count:>5}x  {text!r}")

And `lubotu3\`` is the actual bot — it pastes canned answers when someone says a keyword.
Its repeat rate is 0.386, *below* the threshold that caught `brobostigon`.

Repetition does not separate scripts from people, because people have rituals.

> That greeting returns in lesson 5, where how someone writes stops being noise and becomes
> the thing being measured.